In [112]:
cd /Users/samadsulyman

/Users/samadsulyman


In [113]:
cd /Users/samadsulyman/Desktop/Data Science Project/Raw Data

/Users/samadsulyman/Desktop/Data Science Project/Raw Data


In [114]:
#Data Science Final Project - Abdulsamad Sulyman (AKS2288)
#This notebook is dedicated to the setup of our directories/file folders and the preliminary cleaning of our raw datasets
#INSTALLS
#!pip install rapidfuzz unidecode
#IMPORTS
import os
import sys
import numpy as np
import pandas as pd
from tqdm import tqdm
import glob
from rapidfuzz import fuzz, process
from unidecode import unidecode 
import unicodedata

In [115]:
# Load a single dataset with safe encoding + data type handling
def load_dataset(path):
    try:
        df = pd.read_csv(path, low_memory=False)  # main attempt
        return df
    except UnicodeDecodeError:
        print(f"UTF-8 failed for {path}. Retrying with latin1...")
        df = pd.read_csv(path, encoding="latin1", low_memory=False)
        return df
    except Exception as e:
        print(f"Error loading {path}: {e}")
        return None


# Load all datasets
def load_all_datasets():
    base_path = "/Users/samadsulyman/Desktop/Data Science Project/Raw Data"
    file_names = ["Dataset1.csv", "Dataset2.csv", "Dataset3.csv", "Dataset4.csv"]

    datasets = {}
    for file_name in file_names:
        path = os.path.join(base_path, file_name)
        df = load_dataset(path)
        datasets[file_name.replace(".csv", "")] = df
        if df is not None:
            print(f"Loaded: {file_name} | Shape: {df.shape}")
        else:
            print(f"Failed to load: {file_name}")

    return datasets


# Inspect datasets
def inspect_datasets():
    for name, df in datasets.items():   # uses global 'datasets'
        print("\n" + "="*80)
        print(f"DATASET: {name}")
        print("="*80)

        if df is None:
            print("Could not load this dataset.")
            continue

        print("\nColumns:")
        print(df.columns.tolist())

        print("\nFirst 5 rows:")
        print(df.head())

        print("\nShape:", df.shape)
        print("="*80)


In [116]:
datasets = load_all_datasets()
inspect_datasets()

Loaded: Dataset1.csv | Shape: (8599212, 7)
Loaded: Dataset2.csv | Shape: (2906327, 8)
UTF-8 failed for /Users/samadsulyman/Desktop/Data Science Project/Raw Data/Dataset3.csv. Retrying with latin1...
Loaded: Dataset3.csv | Shape: (2513868, 11)
Loaded: Dataset4.csv | Shape: (27635763, 15)

DATASET: Dataset1

Columns:
['dt', 'AverageTemperature', 'AverageTemperatureUncertainty', 'City', 'Country', 'Latitude', 'Longitude']

First 5 rows:
           dt  AverageTemperature  AverageTemperatureUncertainty   City  \
0  1743-11-01               6.068                          1.737  Århus   
1  1743-12-01                 NaN                            NaN  Århus   
2  1744-01-01                 NaN                            NaN  Århus   
3  1744-02-01                 NaN                            NaN  Århus   
4  1744-03-01                 NaN                            NaN  Århus   

   Country Latitude Longitude  
0  Denmark   57.05N    10.33E  
1  Denmark   57.05N    10.33E  
2  Denmark   57

In [138]:
# -------------------------------
# Normalization helper
# -------------------------------
def normalize(s):
    return unidecode(s).lower().strip()

# -------------------------------
# Allowed cities (original list)
# -------------------------------
target_cities = {
    "Cairo", "Lagos", "Nairobi", "Johannesburg", "Addis Ababa",
    "Accra", "Dakar", "Khartoum", "Kinshasa", "Dar es Salaam",
    "Kampala", "Algiers", "Maputo", "Abidjan", "Luanda"
}

# Pre-normalized allowed cities
normalized_cities = {normalize(c) for c in target_cities}


def is_city_allowed(city, threshold=90):
    """
    Robust city matcher with:
    - case insensitivity
    - accent normalization
    - fuzzy near match
    """
    if not isinstance(city, str):
        return False

    norm = normalize(city)

    # Exact normalized match
    if norm in normalized_cities:
        return True

    # Fuzzy search across normalized list
    match, score, _ = process.extractOne(
        norm,
        normalized_cities,
        scorer=fuzz.token_sort_ratio
    )

    return score >= threshold


# -------------------------------
# Filter Dataset1
# -------------------------------
if datasets.get("Dataset1") is not None:
    df1 = datasets["Dataset1"].copy()
    df1["City"] = df1["City"].astype(str)
    df1["City_match"] = df1["City"].apply(is_city_allowed)
    df1_filtered = df1[df1["City_match"] == True]


# -------------------------------
# Filter Dataset2
# -------------------------------
if datasets.get("Dataset2") is not None:
    df2 = datasets["Dataset2"].copy()
    df2["City"] = df2["City"].astype(str)
    df2["City_match"] = df2["City"].apply(is_city_allowed)
    df2_filtered = df2[df2["City_match"] == True]


# -------------------------------
# Filter Dataset3
# -------------------------------
# Allowed areas for Dataset3 (original list stays the same)
allowed_areas = {
    "Algeria", "Angola", "Côte d'Ivoire", "Democratic Republic of the Congo",
    "Egypt", "Ethiopia", "Ghana", "Kenya", "Mozambique", "Nigeria", "Senegal",
    "South Africa", "Sudan", "United Republic of Tanzania", "Uganda",
    "Africa", "Northern Africa", "Western Africa", "Eastern Africa",
    "Middle Africa", "Southern Africa"
}

# Normalize allowed areas (case, accents, spaces)
def normalize(s):
    return unidecode(s).lower().strip()

normalized_allowed = {normalize(a) for a in allowed_areas}


def is_allowed(area, threshold=90):
    """
    Return True if `area` roughly matches any allowed area 
    using normalization + fuzzy matching.
    """
    if not isinstance(area, str):
        return False

    norm = normalize(area)

    # Exact normalized match
    if norm in normalized_allowed:
        return True

    # Fuzzy match with threshold (controls strictness)
    match, score, _ = process.extractOne(
        norm, normalized_allowed, scorer=fuzz.token_sort_ratio
    )

    return score >= threshold


# --- Filter Dataset3 ---
if datasets["Dataset3"] is not None:
    df3 = datasets["Dataset3"].copy()

    df3["Area"] = df3["Area"].astype(str)

    df3["Area_match"] = df3["Area"].apply(is_allowed)
    df3_filtered = df3[df3["Area_match"] == True]

# -------------------------------
# Filter Dataset4
# -------------------------------
if datasets.get("Dataset4") is not None:
    df4 = datasets["Dataset4"].copy()
    df4["city_name"] = df4["city_name"].astype(str)
    df4["City_match"] = df4["city_name"].apply(is_city_allowed)
    df4_filtered = df4[df4["City_match"] == True]

# -------------------------------
# Collect filtered datasets into a dictionary
# -------------------------------
filtered_datasets = {}

for name in ["df1_filtered", "df2_filtered", "df3_filtered", "df4_filtered"]:
    if name in globals():
        filtered_datasets[name] = globals()[name]

# -------------------------------
# Print summary
# -------------------------------
print("\nFiltered datasets collected:")
for name, df in filtered_datasets.items():
    print(f"- {name}: shape={df.shape}")


Filtered Shapes Summary:

Filtered datasets collected:
- df1_filtered: shape=(33133, 8)
- df2_filtered: shape=(90314, 9)
- df3_filtered: shape=(292809, 12)
- df4_filtered: shape=(158144, 16)


In [140]:
def preview_datasets(datasets):
    """
    For each dataset in the dictionary, print the first 5, middle 5, and last 5 rows.
    """
    for name, df in datasets.items():
        if df is None or df.empty:
            print(f"{name} is empty or None.\n")
            continue

        print(f"\n{'='*80}\nDATASET: {name}\n{'='*80}")

        # First 5 rows
        print("\nFirst 5 rows:")
        print(df.head(5))

        # Middle 5 rows
        mid_index = len(df) // 2
        start = max(mid_index - 2, 0)
        end = min(mid_index + 3, len(df))  # 5 rows total
        print("\nMiddle 5 rows:")
        print(df.iloc[start:end])

        # Last 5 rows
        print("\nLast 5 rows:")
        print(df.tail(5))

# --- Usage ---
preview_datasets(filtered_datasets)


DATASET: df1_filtered

First 5 rows:
               dt  AverageTemperature  AverageTemperatureUncertainty     City  \
43553  1849-01-01              26.704                          1.435  Abidjan   
43554  1849-02-01              27.434                          1.362  Abidjan   
43555  1849-03-01              28.101                          1.612  Abidjan   
43556  1849-04-01              26.140                          1.387  Abidjan   
43557  1849-05-01              25.427                          1.200  Abidjan   

             Country Latitude Longitude  City_match  
43553  Côte D'Ivoire    5.63N     3.23W        True  
43554  Côte D'Ivoire    5.63N     3.23W        True  
43555  Côte D'Ivoire    5.63N     3.23W        True  
43556  Côte D'Ivoire    5.63N     3.23W        True  
43557  Côte D'Ivoire    5.63N     3.23W        True  

Middle 5 rows:
                 dt  AverageTemperature  AverageTemperatureUncertainty  \
3502684  1949-11-01              17.639                      

In [142]:
def normalize_text(text):
    """Remove accents, strip whitespace, and title-case a string."""
    if pd.isna(text):
        return text
    text = str(text).strip()
    # Remove accents
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('utf-8')
    # Title case
    return text.title()

def standardize_datasets(datasets):
    """
    Standardizes column names, dates, numeric columns, and city/area names for Dataset1-4.
    Returns a dictionary with standardized dataframes.
    """
    standardized = {}

    for name, df in datasets.items():
        if df is None:
            continue

        df = df.copy()
        # --- Dataset-specific date handling ---
        if name == "Dataset1" and "dt" in df.columns:
            df["date"] = pd.to_datetime(df["dt"], errors="coerce")
            df.drop(columns=["dt"], inplace=True)

        elif name == "Dataset2" and all(c in df.columns for c in ["year", "month", "day"]):
            df["date"] = pd.to_datetime(df[["year", "month", "day"]], errors="coerce")
            df.drop(columns=["year", "month", "day"], inplace=True)

        elif name == "Dataset3" and "year" in df.columns:
            df["date"] = pd.to_datetime(df["year"], format="%Y", errors="coerce")

        elif name == "Dataset4" and "date" in df.columns:
            df["date"] = pd.to_datetime(df["date"], errors="coerce")

        standardized[name] = df

    return standardized

# --- Usage ---
standardized_datasets = standardize_datasets(filtered_datasets)

# Example: check date ranges after standardization
for name, df in standardized_datasets.items():
    if "date" in df.columns:
        print(f"{name} date range: {df['date'].min()} to {df['date'].max()}")

preview_datasets(standardized_datasets)

df4_filtered date range: 1942-09-07 00:00:00.000000 to 2023-09-05 00:00:00.000000

DATASET: df1_filtered

First 5 rows:
               dt  AverageTemperature  AverageTemperatureUncertainty     City  \
43553  1849-01-01              26.704                          1.435  Abidjan   
43554  1849-02-01              27.434                          1.362  Abidjan   
43555  1849-03-01              28.101                          1.612  Abidjan   
43556  1849-04-01              26.140                          1.387  Abidjan   
43557  1849-05-01              25.427                          1.200  Abidjan   

             Country Latitude Longitude  City_match  
43553  Côte D'Ivoire    5.63N     3.23W        True  
43554  Côte D'Ivoire    5.63N     3.23W        True  
43555  Côte D'Ivoire    5.63N     3.23W        True  
43556  Côte D'Ivoire    5.63N     3.23W        True  
43557  Côte D'Ivoire    5.63N     3.23W        True  

Middle 5 rows:
                 dt  AverageTemperature  AverageTempe

In [154]:
def print_date_ranges(datasets):
    """
    Prints the date range (min → max) for each filtered dataset.
    Handles df1_filtered → df4_filtered formats correctly.
    """
    print("\n================ DATE RANGES =================")
    
    for name, df in datasets.items():
        if df is None or df.empty:
            print(f"{name}: empty dataset")
            continue
        
        temp = df.copy()
        date_col = None

        # df1_filtered: dt column
        if "dt" in temp.columns:
            date_col = "dt"
            temp["date"] = pd.to_datetime(temp[date_col], errors="coerce")
        
        # df2_filtered: Year, Month, Day
        elif all(c in temp.columns for c in ["Year", "Month", "Day"]):
            date_col = "Year-Month-Day"
            temp["date"] = pd.to_datetime(temp[["Year", "Month", "Day"]], errors="coerce")
        
        # df3_filtered: Year only
        elif "Year" in temp.columns:
            date_col = "Year"
            temp["date"] = pd.to_datetime(temp["Year"], format="%Y", errors="coerce")
        
        # df4_filtered: date column
        elif "date" in temp.columns:
            date_col = "date"
            temp["date"] = pd.to_datetime(temp[date_col], errors="coerce")
        
        if date_col is None:
            print(f"{name}: no recognizable date column")
            continue
        
        valid_dates = temp["date"].dropna()
        if valid_dates.empty:
            print(f"{name}: no valid dates")
        else:
            print(f"{name}: {valid_dates.min()} to {valid_dates.max()}")

print_date_ranges(standardized_datasets)


================ DATE RANGES =================
df1_filtered: 1753-01-01 00:00:00 to 2013-09-01 00:00:00
df2_filtered: 1995-01-01 00:00:00 to 2020-05-13 00:00:00
df3_filtered: 1961-01-01 00:00:00 to 2019-01-01 00:00:00
df4_filtered: 1942-09-07 00:00:00 to 2023-09-05 00:00:00


In [160]:
cd /Users/samadsulyman/Desktop/Data Science Project/Processed Data

/Users/samadsulyman/Desktop/Data Science Project/Processed Data


In [227]:
# -------------------------------------------------------------
# DATASET 1  (column = AverageTemperature)
# -------------------------------------------------------------
def clean_dataset1(df):
    df = df[df['AverageTemperature'] != -99].copy()
    return df


# -------------------------------------------------------------
# DATASET 2  (column = AvgTemperature)
# -------------------------------------------------------------
def clean_dataset2(df):
    df = df[df['AvgTemperature'] != -99].copy()
    return df


# -------------------------------------------------------------
# DATASET 3  (no temp column to clean unless specified)
# -------------------------------------------------------------
def clean_dataset3(df):
    # No temperature column listed — return unchanged
    return df


# -------------------------------------------------------------
# DATASET 4  (assuming no -99 temps or unspecified)
# -------------------------------------------------------------
def clean_dataset4(df):
    # No temperature column listed — return unchanged
    return df


# =============================================================
# Load → Clean → Save same files (overwrite)
# =============================================================
df1 = pd.read_csv("ProcessedDataset1.csv")
df2 = pd.read_csv("ProcessedDataset2.csv")
df3 = pd.read_csv("ProcessedDataset3.csv")
df4 = pd.read_csv("ProcessedDataset4.csv", low_memory=False)

df1_clean = clean_dataset1(df1)
df2_clean = clean_dataset2(df2)
df3_clean = clean_dataset3(df3)
df4_clean = clean_dataset4(df4)

df1_clean.to_csv("ProcessedDataset1.csv", index=False)
df2_clean.to_csv("ProcessedDataset2.csv", index=False)
df3_clean.to_csv("ProcessedDataset3.csv", index=False)
df4_clean.to_csv("ProcessedDataset4.csv", index=False)

print("✓ Removed -99 temperature values where applicable and saved datasets.")

✓ Removed -99 temperature values where applicable and saved datasets.
